In [1]:
import os

import pandas as pd
from tqdm import tqdm

from PIL import Image
from typing import Optional, Tuple, Union

import torch
import torch.nn as nn
from typing import Optional, Tuple, Union
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

In [2]:
class ImageDataset(Dataset):
    """
    Кастомный датасет для изображений с маркировкой (0 — белый фон, 1 — цветной фон).

    Поддерживает как размеченные данные (train), так и тестовые (без target).
    При инициализации автоматически фильтрует отсутствующие изображения.

    Args:
        df (pd.DataFrame): Таблица с колонкой 'id' (название файла) и 'target' (метка, если labeled=True).
        img_root (str): Путь к директории с изображениями.
                        Для train: ожидается структура /root/target/id.jpg
                        Для test:  просто /root/id.jpg
        transform (transforms.Compose, optional): Аугментации / предобработка. По умолчанию None.
        labeled (bool): True — если train, False — если test. По умолчанию True.
    """

    def __init__(self, df: pd.DataFrame, img_root: str, transform: Optional[transforms.Compose] = None, labeled: bool = True):
        self.df = df.copy()
        self.img_root = img_root
        self.transform = transform
        self.labeled = labeled

        keep_rows = []
        for _, row in tqdm(self.df.iterrows(), total=len(self.df),
                           desc="Проверка файлов" + (" (train)" if self.labeled else " (test)")):
            img_id = row["id"]
            label = row["target"] if self.labeled else None
            img_file = img_id if img_id.endswith(".jpg") else f"{img_id}.jpg"

            if self.labeled:
                img_path = os.path.join(self.img_root, str(label), img_file)
            else:
                img_path = os.path.join(self.img_root, img_file)

            if os.path.exists(img_path):
                keep_rows.append(True)
            else:
                keep_rows.append(False)

        self.df = self.df[keep_rows].reset_index(drop=True)

    def __len__(self) -> int:
        """
        Возвращает количество доступных изображений в датасете.

        Returns:
            int: число валидных записей
        """
        return len(self.df)

    def __getitem__(self, idx: int) -> Union[Tuple[torch.Tensor, int], Tuple[torch.Tensor, str]]:
        """
        Загружает и возвращает изображение и его метку (или имя) по индексу.

        Args:
            idx (int): индекс изображения в датафрейме

        Returns:
            Tuple[torch.Tensor, int]  — для labeled=True (train)
            Tuple[torch.Tensor, str]  — для labeled=False (test)
        """
        row = self.df.iloc[idx]
        img_id = row["id"]
        label = row["target"] if self.labeled else -1

        img_file = img_id if img_id.endswith(".jpg") else f"{img_id}.jpg"
        if self.labeled:
            img_path = os.path.join(self.img_root, str(label), img_file)
        else:
            img_path = os.path.join(self.img_root, img_file)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return (image, label) if self.labeled else (image, img_id)


In [3]:
# Пути
train_csv = "/kaggle/input/ghdxfgh/train .csv"
test_csv = "/kaggle/input/ghdxfgh/test.csv"
train_path = "/kaggle/input/ghdxfgh/train/train"
test_path = "/kaggle/input/ghdxfgh/test/test"

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

In [5]:
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

In [6]:
train_dataset = ImageDataset(train_df, img_root=train_path, transform=transform)
test_dataset = ImageDataset(test_df, img_root=test_path, transform=transform, labeled=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Проверка файлов (test): 100%|██████████| 2064/2064 [00:06<00:00, 340.75it/s]


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 1)  # Бинарка
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Обучение (3 эпохи для начала)
for epoch in range(4):
    model.train()
    running_loss = 0
    for imgs, labels in tqdm(train_loader):
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {running_loss:.4f}")

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 158MB/s] 
100%|██████████| 211/211 [02:18<00:00,  1.52it/s]


Epoch 1: Loss = 47.6312


100%|██████████| 211/211 [01:43<00:00,  2.03it/s]


Epoch 2: Loss = 20.2809


100%|██████████| 211/211 [01:44<00:00,  2.02it/s]


Epoch 3: Loss = 10.0668


100%|██████████| 211/211 [01:45<00:00,  2.00it/s]

Epoch 4: Loss = 4.9663


In [8]:
model.eval()
predictions = []
ids = []

with torch.no_grad():
    for imgs, img_ids in tqdm(test_loader):
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.sigmoid(outputs).cpu().numpy()
        preds = (probs > 0.5).astype(int).flatten()
        predictions.extend(preds)
        ids.extend(img_ids)

submission = pd.DataFrame({
    "id": ids,
    "target": predictions
})

submission = submission.sort_values("id")  # на всякий случай
submission.to_csv("submission.csv", index=False)


100%|██████████| 64/64 [00:47<00:00,  1.36it/s]
